# 智能体

将语言模型与工具相结合，创建能够对任务进行推理、决定使用哪些工具并迭代地寻求解决方案的系统。

create_agent提供可用于生产环境的代理实现。


LLM 智能体循环运行各种工具以实现目标。智能体持续运行，直到满足停止条件为止——即模型发出最终输出或达到迭代次数限制。

<img src="https://i-blog.csdnimg.cn/direct/4b5a4f8b85854dfe83a3c68af1922b30.png" alt="agent" style="zoom:50%;" />

`create_agent`使用LangGraph构建基于图的代理运行时。图由节点（步骤）和边（连接）组成，定义了代理如何处理信息。代理在这个图中移动，执行诸如模型节点（调用模型）、工具节点（执行工具）或中间件之类的节点。

## 核心组件
​
### 模型
模型是智能体的推理引擎。它可以通过多种方式进行指定，支持静态和动态模型选择。
​
### 静态模型
静态模型在创建代理时配置一次，并在整个执行过程中保持不变。这是最常见、最直接的方法。
要从以下位置初始化静态模型模型标识符字符串：



In [ ]:
tools = []

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    "gpt-5",
    tools=tools
)

为了更好地控制模型配置，可以直接使用提供程序包初始化模型实例。在本例中，我们使用 `<path>` ChatOpenAI。有关其他可用的聊天模型类，请参阅“聊天模型”部分。

In [ ]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="gpt-5",
    temperature=0.1,
    max_tokens=1000,
    timeout=30
    # ... (other params)
)
agent = create_agent(model, tools=tools)


模型实例让您可以完全控制配置。当您需要设置特定参数（例如`temperature`、`max_tokens` `timeouts`、`base_url` 以及其他特定于提供程序的设置）时，请使用它们。请参阅参考文档以了解模型中可用的参数和方法。
​


### 动态模型


动态模型是在运行时根据当前状态和上下文选择的。 这可以实现复杂的路由逻辑和成本优化。


要使用动态模型，请使用 `@wrap_model_call` 装饰器创建中间件，该装饰器会修改请求中的模型：



In [ ]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse


basic_model = ChatOpenAI(model="gpt-4o-mini")
advanced_model = ChatOpenAI(model="gpt-4o")

@wrap_model_call
def dynamic_model_selection(request: ModelRequest, handler) -> ModelResponse:
    """Choose model based on conversation complexity."""
    message_count = len(request.state["messages"])

    if message_count > 10:
        # Use an advanced model for longer conversations
        model = advanced_model
    else:
        model = basic_model

    request.model = model
    return handler(request)

agent = create_agent(
    model=basic_model,  # Default model
    tools=tools,
    middleware=[dynamic_model_selection]
)

> 使用结构化输出时，不支持预绑定模型（bind_tools已调用过的模型）。如果需要使用结构化输出进行动态模型选择，请确保传递给中间件的模型未预先绑定。

> 有关模型配置的详细信息，请参阅“模型”部分。有关动态模型选择模式，请参阅中间件中的“动态模型”部分。

### 工具
工具赋予智能体执行操作的能力。智能体超越了简单的仅模型工具绑定，还能实现以下功能：
- 连续调用多个工具（由单个提示触发）
- 在适当的时候并行调用工具
- 基于先前结果的动态工具选择
- 工具重试逻辑和错误处理
- 跨工具调用保持状态持久性

有关更多信息，请参阅“工具”部分。

#### 定义工具
将工具列表传递给代理人。




In [ ]:
from langchain.tools import tool
from langchain.agents import create_agent


@tool
def search(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"

@tool
def get_weather(location: str) -> str:
    """Get weather information for a location."""
    return f"Weather in {location}: Sunny, 72°F"

agent = create_agent(model, tools=[search, get_weather])

如果提供的工具列表为空，则代理将由一个没有工具调用功能的 LLM 节点组成。

##### 工具错误处理
要自定义工具错误的处理方式，请使用[@wrap_tool_call](https://reference.langchain.com/python/langchain/middleware/?_gl=1*14i1taj*_gcl_au*MzA2ODMwODE2LjE3NjE5MTM5NzE.*_ga*MTEyMjY1OTA5MS4xNzUzNDI3NzE4*_ga_47WX3HKKY2*czE3NjIzMDM5NzUkbzgwJGcxJHQxNzYyMzEzMTE1JGo2MCRsMCRoMA..#langchain.agents.middleware.wrap_tool_call)装饰器创建中间件：





In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_tool_call
from langchain_core.messages import ToolMessage


@wrap_tool_call
def handle_tool_errors(request, handler):
    """Handle tool execution errors with custom messages."""
    try:
        return handler(request)
    except Exception as e:
        # Return a custom error message to the model
        return ToolMessage(
            content=f"Tool error: Please check your input and try again. ({str(e)})",
            tool_call_id=request.tool_call["id"]
        )

agent = create_agent(
    model="gpt-4o",
    tools=[search, get_weather],
    middleware=[handle_tool_errors]
)

[ToolMessage](https://reference.langchain.com/python/langchain/messages/?_gl=1*1yysq7v*_gcl_au*MzA2ODMwODE2LjE3NjE5MTM5NzE.*_ga*MTEyMjY1OTA5MS4xNzUzNDI3NzE4*_ga_47WX3HKKY2*czE3NjIzMDM5NzUkbzgwJGcxJHQxNzYyMzEzMTE1JGo2MCRsMCRoMA..#langchain.messages.ToolMessage)当工具运行失败时，代理将返回带有自定义错误消息的响应：

```python
[

    ToolMessage(
        content="Tool error: Please check your input and try again. (division by zero)",
        tool_call_id="..."
    ),

]
```

##### ReAct循环中的工具使用

智能体遵循 ReAct（“推理 + 行动”）模式，在简短的推理步骤和有针对性的工具调用之间交替进行，并将由此产生的观察结果输入到后续决策中，直到能够给出最终答案。



React循环示例

提示：确定目前最流行的无线耳机并确认其供货情况。

================================ Human Message =================================

查找目前最流行的无线耳机并检查是否有库存

- Reasoning：“受欢迎程度是有时间限制的，我需要使用提供的搜索工具。”        
- action：执行 search_products("wireless headphones")         

================================== Ai Message ==================================
Tool Calls:            
  search_products (call_abc123)        
 Call ID: call_abc123         
  Args:          
    query: wireless headphones         
 

================================= Tool Message =================================

找到 5 款与“无线耳机”相匹配的产品。前 5 名成绩：WH-1000XM5、

- Reasoning: “我需要在回答问题之前确认排名第一的商品是否有货”
- Acting: Call check_inventory("WH-1000XM5")

================================== Ai Message ==================================
Tool Calls:        
  check_inventory (call_def456)     
 Call ID: call_def456    
  Args:    
    product_id: WH-1000XM5        


================================= Tool Message =================================

产品 WH-1000XM5：库存 10 台

Reasoning：“我掌握了最热门的型号及其库存状态。我现在可以回答用户的问题了。”
Acting：给出最终答案


================================== Ai Message ==================================

我发现无线耳机（型号 WH-1000XM5）有 10 台库存......


> 要了解有关工具的更多信息，请参阅“工具”部分。

#### 系统提示
您可以通过提供提示来控制代理处理任务的方式。该`system_prompt`参数可以以字符串形式提供：




In [ ]:
agent = create_agent(
    model,
    tools,
    system_prompt="You are a helpful assistant. Be concise and accurate."
)

如果没有`system_prompt`提供任务，代理将直接从消息中推断其任务。

##### 动态系统提示
对于需要根据运行时上下文或代理状态修改系统提示的更高级用例，可以使用中间件。
装饰@dynamic_prompt器会创建中间件，该中间件会根据模型请求动态生成系统提示：

In [ ]:
from typing import TypedDict

from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt, ModelRequest


class Context(TypedDict):
    user_role: str

@dynamic_prompt
def user_role_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on user role."""
    user_role = request.runtime.context.get("user_role", "user")
    base_prompt = "You are a helpful assistant."

    if user_role == "expert":
        return f"{base_prompt} Provide detailed technical responses."
    elif user_role == "beginner":
        return f"{base_prompt} Explain concepts simply and avoid jargon."

    return base_prompt

agent = create_agent(
    model="gpt-4o",
    tools=[web_search],
    middleware=[user_role_prompt],
    context_schema=Context
)

# The system prompt will be set dynamically based on context
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Explain machine learning"}]},
    context={"user_role": "expert"}
)

> 有关消息类型和格式的更多详细信息，请参阅“消息”部分。有关完整的中间件文档，请参阅“[中间件](https://docs.langchain.com/oss/python/langchain/middleware)”部分。

### 调用
您可以通过向代理传递更新来调用它State。所有代理的状态中都包含一系列消息；要调用代理，请传递一条新消息：

```json
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's the weather in San Francisco?"}]}
)
```

有关从代理流式传输步骤和/或令牌的信息，请参阅流式传输指南。

否则，该代理遵循 LangGraph Graph API并支持所有相关方法。
​


## 高级概念
​
### 结构化输出
在某些情况下，您可能希望代理以特定格式返回输出。LangChain 通过`response_format`参数提供了结构化输出的策略。

#### 工具策略
`ToolStrategy`使用人工工具调用来生成结构化输出。这适用于任何支持工具调用的模型：



In [ ]:
from pydantic import BaseModel
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy


class ContactInfo(BaseModel):
    name: str
    email: str
    phone: str

agent = create_agent(
    model="gpt-4o-mini",
    tools=[search_tool],
    response_format=ToolStrategy(ContactInfo)
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]
# ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

#### 提供者策略
ProviderStrategy使用模型提供商的原生结构化输出生成功能。这种方式更可靠，但仅适用于支持原生结构化输出的提供商（例如 OpenAI）：

In [ ]:
from langchain.agents.structured_output import ProviderStrategy

agent = create_agent(
    model="gpt-4o",
    response_format=ProviderStrategy(ContactInfo)
) 

### 记忆
客服人员会通过消息状态自动维护对话历史记录。您还可以配置客服人员使用自定义状态方案，以便在对话过程中记住其他信息。

存储在状态中的信息可以被视为智能体的短期记忆：

自定义状态模式必须扩展`AgentState`为`TypedDict`。

定义自定义状态有两种方法：
- 通过中间件（首选）
- 通过state_schema​create_agent

> state_schema通过中间件定义自定义状态比通过on定义自定义状态更可取create_agent，因为它允许你将状态扩展在概念上限制在相关的中间件和工具范围内。
> state_schema为了向后兼容，仍然支持create_agent。

#### 通过中间件定义状态
当您的自定义状态需要被附加到该中间件的特定中间件钩子和工具访问时，可以使用中间件来定义自定义状态。



In [ ]:
from langchain.agents import AgentState
from langchain.agents.middleware import AgentMiddleware


class CustomState(AgentState):
    user_preferences: dict

class CustomMiddleware(AgentMiddleware):
    state_schema = CustomState
    tools = [tool1, tool2]

    def before_model(self, state: CustomState, runtime) -> dict[str, Any] | None:
        ...

agent = create_agent(
    model,
    tools=tools,
    middleware=[CustomMiddleware()]
)

# The agent can now track additional state beyond messages
result = agent.invoke({
    "messages": [{"role": "user", "content": "I prefer technical explanations"}],
    "user_preferences": {"style": "technical", "verbosity": "detailed"},
})

#### 通过以下方式定义状态state_schema
使用该`state_schema`参数作为快捷方式，定义仅在工具中使用的自定义状态。




In [ ]:
from langchain.agents import AgentState


class CustomState(AgentState):
    user_preferences: dict

agent = create_agent(
    model,
    tools=[tool1, tool2],
    state_schema=CustomState
)
# The agent can now track additional state beyond messages
result = agent.invoke({
    "messages": [{"role": "user", "content": "I prefer technical explanations"}],
    "user_preferences": {"style": "technical", "verbosity": "detailed"},
})

> 从现在起langchain 1.0，自定义状态模式必须是TypedDict类型。不再支持 Pydantic 模型和数据类。更多详情请参阅v1 迁移指南。


### 流媒体
我们已经了解了如何调用代理程序invoke来获取最终响应。如果代理程序执行多个步骤，这可能需要一些时间。为了显示中间进度，我们可以实时回传消息。



In [ ]:
for chunk in agent.stream({
    "messages": [{"role": "user", "content": "Search for AI news and summarize the findings"}]
}, stream_mode="values"):
    # Each chunk contains the full state at that point
    latest_message = chunk["messages"][-1]
    if latest_message.content:
        print(f"Agent: {latest_message.content}")
    elif latest_message.tool_calls:
        print(f"Calling tools: {[tc['name'] for tc in latest_message.tool_calls]}")

### 中间件
中间件提供了强大的扩展性，可用于在执行的不同阶段自定义代理行为。您可以使用中间件来：

在调用模型之前处理状态（例如，消息修剪、上下文注入）
- 修改或验证模型的响应（例如，防护措施、内容过滤）
- 使用自定义逻辑处理工具执行错误
- 基于状态或上下文实现动态模型选择
- 添加自定义日志记录、监控或分析功能

中间件可以无缝集成到代理的执行图中，使您能够在关键点拦截和修改数据流，而无需更改核心代理逻辑​​。